# Prints dimensionality and triplet accuracy with and without signifficance criterion

Reads and collects dimensionalities from `.json` files for each observer / task / image-set

In [1]:
import pandas as pd
import numpy as np

from io_json import read_json_dataclass, read_json_dict


In [2]:
obs = 'O1'
task = 'classic'
image_set = 'greebles'
dim_result = read_json_dataclass(f"../data/triplet_accuracy_vs_dimensionality-{obs}-{task}-{image_set}.json")
idx = np.where(dim_result.dimensions==dim_result.estimated_dimension)
assert(len(idx)==1)
test_scores = dim_result.test_scores[idx[0], :]

In [3]:
np.mean(test_scores)

0.821027339317033

In [4]:
def read_data(criterion):

    observers = ['O1', 'O2', 'O3', 'O4', 'O5', 'pooled O1 - O5', 'P1', 'P2']
    tasks = ['classic', 'odd one out']
    imsets = ['action', 'greebles', 'taxonomic']
    
    data = []
    for obs in observers:
        for task in tasks:
            for image_set in imsets:
                # read dim_result from JSON, this is the dataclass returned by cblearn
                dim_result = read_json_dataclass(f"../data/triplet_accuracy_vs_dimensionality-{obs}-{task}-{image_set}.json")
                
                if criterion is None:                    
                    dim = dim_result.estimated_dimension
                elif criterion == 'LogOdds':
                    fname = f"../data/triplet_accuracy_vs_dimensionality-{obs}-{task}-{image_set}_LogOdds.json"
                    dim_result_recalculated = read_json_dict(fname)                
                    dim = dim_result_recalculated['estimated_dimension']

                # get triplet accuracy for the choosen dimension
                test_scores = dim_result.test_scores[dim-1, :]
                triplet_acc = np.mean(test_scores)*100
                  
                #obs_label = 'Pooled O1-O5' if (obs=='allobs') else obs
                data.append([obs, task, image_set, dim, triplet_acc])
                
    df = pd.DataFrame(data, columns = ['obs', 'task', 'image_set', 'dimensionality', 'triplet accuracy'])

    return df

## With only signifficance criterion

In [5]:
df = read_data(criterion=None)
df.pivot_table(values='dimensionality', index='obs', columns = ['image_set', 'task'])

image_set       action             greebles             taxonomic            
task           classic odd one out  classic odd one out   classic odd one out
obs                                                                          
O1                 2.0         2.0      2.0         1.0       3.0         2.0
O2                 1.0         1.0      2.0         2.0       2.0         2.0
O3                 2.0         3.0      1.0         1.0       2.0         1.0
O4                 2.0         2.0      2.0         2.0       1.0         1.0
O5                 1.0         1.0      3.0         2.0       2.0         1.0
P1                 2.0         2.0      2.0         1.0       2.0         1.0
P2                 2.0         1.0      2.0         1.0       3.0         3.0
pooled O1 - O5     4.0         3.0      2.0         2.0       2.0         2.0

In [6]:
df.pivot_table(values='triplet accuracy', index='obs', columns = ['image_set', 'task']).round(1)

image_set       action             greebles             taxonomic            
task           classic odd one out  classic odd one out   classic odd one out
obs                                                                          
O1                87.1        96.3     82.1        90.0      85.6        95.7
O2                86.8        96.1     89.9        98.4      93.2        97.3
O3                80.3        83.6     69.3        76.7      86.2        91.3
O4                80.4        91.8     77.3        89.9      75.8        95.1
O5                91.2        99.4     81.6        90.2      94.7        99.7
P1                91.4        98.6     77.0        84.2      92.8        99.7
P2                81.6        85.0     83.1        90.6      86.8        92.8
pooled O1 - O5    85.6        89.7     75.5        82.0      83.5        91.8

In [7]:
df.pivot_table(values='triplet accuracy', index='obs', columns = ['task']).round(1)

task,classic,odd one out
obs,,
O1,84.9,94.0
O2,90.0,97.3
O3,78.6,83.9
O4,77.8,92.2
O5,89.2,96.4
P1,87.1,94.2
P2,83.8,89.5
pooled O1 - O5,81.5,87.8


In [8]:
## single table with triplet accuracy per imageset and mean across dataset

In [9]:
def triplet_accuracy_table(dataframe, decimals=1):
    """Single table with triplet accuracy per image set plus the mean across image sets.

    Rows are observers, columns are a (image_set, task) MultiIndex where the last
    image_set block is 'mean' and holds the average across the image sets.
    """
    per_imageset = dataframe.pivot_table(
        values='triplet accuracy', index='obs', columns=['image_set', 'task']
    )
    across_imagesets = dataframe.pivot_table(
        values='triplet accuracy', index='obs', columns=['task']
    )
    across_imagesets.columns = pd.MultiIndex.from_product(
        [['mean'], across_imagesets.columns], names=per_imageset.columns.names
    )

    table = pd.concat([per_imageset, across_imagesets], axis=1).round(decimals)
    return table


table = triplet_accuracy_table(df)
table

image_set       action             greebles             taxonomic              \
task           classic odd one out  classic odd one out   classic odd one out   
obs                                                                             
O1                87.1        96.3     82.1        90.0      85.6        95.7   
O2                86.8        96.1     89.9        98.4      93.2        97.3   
O3                80.3        83.6     69.3        76.7      86.2        91.3   
O4                80.4        91.8     77.3        89.9      75.8        95.1   
O5                91.2        99.4     81.6        90.2      94.7        99.7   
P1                91.4        98.6     77.0        84.2      92.8        99.7   
P2                81.6        85.0     83.1        90.6      86.8        92.8   
pooled O1 - O5    85.6        89.7     75.5        82.0      83.5        91.8   

image_set         mean              
task           classic odd one out  
obs                                 
O1                84.9        94.0  
O2                90.0        97.3  
O3                78.6        83.9  
O4                77.8        92.2  
O5                89.2        96.4  
P1                87.1        94.2  
P2                83.8        89.5  
pooled O1 - O5    81.5        87.8

In [10]:
print(table.to_latex(
    float_format='%.1f',
    multicolumn=True,
    multicolumn_format='c',
    caption='Triplet accuracy (percent) per image set and averaged across image sets.',
    label='tab:triplet_accuracy',
))

\begin{table}
\caption{Triplet accuracy (percent) per image set and averaged across image sets.}
\label{tab:triplet_accuracy}
\begin{tabular}{lrrrrrrrr}
\toprule
image_set & \multicolumn{2}{c}{action} & \multicolumn{2}{c}{greebles} & \multicolumn{2}{c}{taxonomic} & \multicolumn{2}{c}{mean} \\
task & classic & odd one out & classic & odd one out & classic & odd one out & classic & odd one out \\
obs &  &  &  &  &  &  &  &  \\
\midrule
O1 & 87.1 & 96.3 & 82.1 & 90.0 & 85.6 & 95.7 & 84.9 & 94.0 \\
O2 & 86.8 & 96.1 & 89.9 & 98.4 & 93.2 & 97.3 & 90.0 & 97.3 \\
O3 & 80.3 & 83.6 & 69.3 & 76.7 & 86.2 & 91.3 & 78.6 & 83.9 \\
O4 & 80.4 & 91.8 & 77.3 & 89.9 & 75.8 & 95.1 & 77.8 & 92.2 \\
O5 & 91.2 & 99.4 & 81.6 & 90.2 & 94.7 & 99.7 & 89.2 & 96.4 \\
P1 & 91.4 & 98.6 & 77.0 & 84.2 & 92.8 & 99.7 & 87.1 & 94.2 \\
P2 & 81.6 & 85.0 & 83.1 & 90.6 & 86.8 & 92.8 & 83.8 & 89.5 \\
pooled O1 - O5 & 85.6 & 89.7 & 75.5 & 82.0 & 83.5 & 91.8 & 81.5 & 87.8 \\
\bottomrule
\end{tabular}
\end{table}



## With signifficance AND effect size criterion

In [11]:
df2 = read_data(criterion='LogOdds')
df2.pivot_table(values='dimensionality', index='obs', columns = ['image_set', 'task'])

image_set       action             greebles             taxonomic            
task           classic odd one out  classic odd one out   classic odd one out
obs                                                                          
O1                 2.0         2.0      2.0         1.0       3.0         2.0
O2                 1.0         1.0      2.0         2.0       2.0         2.0
O3                 2.0         2.0      1.0         1.0       2.0         1.0
O4                 2.0         2.0      2.0         1.0       1.0         1.0
O5                 1.0         1.0      3.0         2.0       2.0         1.0
P1                 2.0         2.0      2.0         1.0       2.0         1.0
P2                 2.0         1.0      2.0         1.0       3.0         3.0
pooled O1 - O5     1.0         1.0      2.0         2.0       2.0         1.0

In [12]:
df2.pivot_table(values='triplet accuracy', index='obs', columns = ['image_set', 'task']).round(1)

image_set       action             greebles             taxonomic            
task           classic odd one out  classic odd one out   classic odd one out
obs                                                                          
O1                87.1        96.3     82.1        90.0      85.6        95.7
O2                86.8        96.1     89.9        98.4      93.2        97.3
O3                80.3        82.1     69.3        76.7      86.2        91.3
O4                80.4        91.8     77.3        89.0      75.8        95.1
O5                91.2        99.4     81.6        90.2      94.7        99.7
P1                91.4        98.6     77.0        84.2      92.8        99.7
P2                81.6        85.0     83.1        90.6      86.8        92.8
pooled O1 - O5    83.3        88.4     75.5        82.0      83.5        91.2

In [13]:
df2.pivot_table(values='triplet accuracy', index='obs', columns = ['task']).round(1)

task,classic,odd one out
obs,,
O1,84.9,94.0
O2,90.0,97.3
O3,78.6,83.4
O4,77.8,91.9
O5,89.2,96.4
P1,87.1,94.2
P2,83.8,89.5
pooled O1 - O5,80.8,87.2


## single table with triplet accuracy per imageset and mean across image sets

In [14]:
table2 = triplet_accuracy_table(df2)
table2

image_set       action             greebles             taxonomic              \
task           classic odd one out  classic odd one out   classic odd one out   
obs                                                                             
O1                87.1        96.3     82.1        90.0      85.6        95.7   
O2                86.8        96.1     89.9        98.4      93.2        97.3   
O3                80.3        82.1     69.3        76.7      86.2        91.3   
O4                80.4        91.8     77.3        89.0      75.8        95.1   
O5                91.2        99.4     81.6        90.2      94.7        99.7   
P1                91.4        98.6     77.0        84.2      92.8        99.7   
P2                81.6        85.0     83.1        90.6      86.8        92.8   
pooled O1 - O5    83.3        88.4     75.5        82.0      83.5        91.2   

image_set         mean              
task           classic odd one out  
obs                                 
O1                84.9        94.0  
O2                90.0        97.3  
O3                78.6        83.4  
O4                77.8        91.9  
O5                89.2        96.4  
P1                87.1        94.2  
P2                83.8        89.5  
pooled O1 - O5    80.8        87.2

In [15]:
print(table2.to_latex(
    float_format='%.1f',
    multicolumn=True,
    multicolumn_format='c',
    caption='Triplet accuracy (percent) per image set and averaged across image sets, '
            'using the significance and effect size criterion.',
    label='tab:triplet_accuracy_logodds',
))

\begin{table}
\caption{Triplet accuracy (percent) per image set and averaged across image sets, using the significance and effect size criterion.}
\label{tab:triplet_accuracy_logodds}
\begin{tabular}{lrrrrrrrr}
\toprule
image_set & \multicolumn{2}{c}{action} & \multicolumn{2}{c}{greebles} & \multicolumn{2}{c}{taxonomic} & \multicolumn{2}{c}{mean} \\
task & classic & odd one out & classic & odd one out & classic & odd one out & classic & odd one out \\
obs &  &  &  &  &  &  &  &  \\
\midrule
O1 & 87.1 & 96.3 & 82.1 & 90.0 & 85.6 & 95.7 & 84.9 & 94.0 \\
O2 & 86.8 & 96.1 & 89.9 & 98.4 & 93.2 & 97.3 & 90.0 & 97.3 \\
O3 & 80.3 & 82.1 & 69.3 & 76.7 & 86.2 & 91.3 & 78.6 & 83.4 \\
O4 & 80.4 & 91.8 & 77.3 & 89.0 & 75.8 & 95.1 & 77.8 & 91.9 \\
O5 & 91.2 & 99.4 & 81.6 & 90.2 & 94.7 & 99.7 & 89.2 & 96.4 \\
P1 & 91.4 & 98.6 & 77.0 & 84.2 & 92.8 & 99.7 & 87.1 & 94.2 \\
P2 & 81.6 & 85.0 & 83.1 & 90.6 & 86.8 & 92.8 & 83.8 & 89.5 \\
pooled O1 - O5 & 83.3 & 88.4 & 75.5 & 82.0 & 83.5 & 91.2 & 80.8 & 87.2

## Combined table: dimensionality and triplet accuracy

One table holding both measures, for export to LaTeX. Uses the significance criterion only.
Rows are the individual observers O1 - O5, followed by the average over them and by the pooled
O1 - O5 data. Columns are split by image set and then by task, with the average across image
sets as the last image-set block.

Each cell reads `dimensionality (triplet accuracy)`, accuracy in percent. Quantities that are
averages (the average row, the mean image-set block) are shown with one decimal, single estimates
as integers.

In [16]:
EXP1_OBSERVERS = ['O1', 'O2', 'O3', 'O4', 'O5']
EXP2_OBSERVERS = ['P1', 'P2']
POOLED_OBSERVER = 'pooled O1 - O5'
AVERAGE_OBSERVER_LABEL = 'average O1 - O5'
MEAN_IMAGE_SET_LABEL = 'mean'
TASKS = ['classic', 'odd one out']
IMAGE_SETS = ['action', 'greebles', 'taxonomic']
MEASURES = ['dimensionality', 'triplet accuracy']

ROW_ORDER = EXP1_OBSERVERS + [AVERAGE_OBSERVER_LABEL, POOLED_OBSERVER] + EXP2_OBSERVERS


def summary_table(dataframe):
    """Dimensionality and triplet accuracy in one numeric table.

    Rows: observers O1 - O5, their average, the pooled O1 - O5 data, then P1 and P2.
    Columns: (image_set, task, measure), with an extra 'mean' image set averaging across
    the three image sets.
    """
    long = dataframe.copy()

    # average across image sets, added as an extra image set
    across_imagesets = long.groupby(['obs', 'task'], as_index=False)[MEASURES].mean()
    across_imagesets['image_set'] = MEAN_IMAGE_SET_LABEL
    long = pd.concat([long, across_imagesets], ignore_index=True)

    # average across observers O1 - O5 only, added as an extra observer
    across_observers = (long[long['obs'].isin(EXP1_OBSERVERS)]
                        .groupby(['task', 'image_set'], as_index=False)[MEASURES].mean())
    across_observers['obs'] = AVERAGE_OBSERVER_LABEL
    long = pd.concat([long, across_observers], ignore_index=True)

    table = long.pivot_table(index='obs', columns=['image_set', 'task'], values=MEASURES)
    table = table.reorder_levels(['image_set', 'task', None], axis=1)
    table = table.reindex(columns=pd.MultiIndex.from_product(
        [IMAGE_SETS + [MEAN_IMAGE_SET_LABEL], TASKS, MEASURES]))
    table = table.reindex(index=ROW_ORDER)
    table.columns.names = ['image_set', 'task', 'measure']
    table.index.name = 'observer'
    return table


def format_summary_table(table):
    """Collapse the two measures into one 'dimensionality (accuracy)' column per cell."""
    columns = pd.MultiIndex.from_product(
        [IMAGE_SETS + [MEAN_IMAGE_SET_LABEL], TASKS], names=['image_set', 'task'])
    formatted = pd.DataFrame(index=table.index, columns=columns, dtype=object)

    for image_set in IMAGE_SETS + [MEAN_IMAGE_SET_LABEL]:
        for task in TASKS:
            dimension = table[(image_set, task, 'dimensionality')]
            accuracy = table[(image_set, task, 'triplet accuracy')]
            # averaged quantities keep one decimal, single estimates stay integer
            is_averaged = pd.Series(
                [(image_set == MEAN_IMAGE_SET_LABEL) or (obs == AVERAGE_OBSERVER_LABEL)
                 for obs in table.index], index=table.index)
            dimension_str = dimension.map('{:.1f}'.format).where(
                is_averaged, dimension.map('{:.0f}'.format))
            formatted[(image_set, task)] = dimension_str + ' (' + accuracy.map('{:.1f}'.format) + ')'

    formatted.index.name = 'observer'
    return formatted


# df holds the significance criterion only
summary = summary_table(df)
summary.round(1)

image_set               action                                  \
task                   classic                     odd one out   
measure         dimensionality triplet accuracy dimensionality   
observer                                                         
O1                         2.0             87.1            2.0   
O2                         1.0             86.8            1.0   
O3                         2.0             80.3            3.0   
O4                         2.0             80.4            2.0   
O5                         1.0             91.2            1.0   
average O1 - O5            1.6             85.1            1.8   
pooled O1 - O5             4.0             85.6            3.0   
P1                         2.0             91.4            2.0   
P2                         2.0             81.6            1.0   

image_set                              greebles                   \
task                                    classic                    
measure         triplet accuracy dimensionality triplet accuracy   
observer                                                           
O1                          96.3            2.0             82.1   
O2                          96.1            2.0             89.9   
O3                          83.6            1.0             69.3   
O4                          91.8            2.0             77.3   
O5                          99.4            3.0             81.6   
average O1 - O5             93.4            2.0             80.0   
pooled O1 - O5              89.7            2.0             75.5   
P1                          98.6            2.0             77.0   
P2                          85.0            2.0             83.1   

image_set                                            taxonomic  \
task               odd one out                         classic   
measure         dimensionality triplet accuracy dimensionality   
observer                                                         
O1                         1.0             90.0            3.0   
O2                         2.0             98.4            2.0   
O3                         1.0             76.7            2.0   
O4                         2.0             89.9            1.0   
O5                         2.0             90.2            2.0   
average O1 - O5            1.6             89.0            2.0   
pooled O1 - O5             2.0             82.0            2.0   
P1                         1.0             84.2            2.0   
P2                         1.0             90.6            3.0   

image_set                                                         \
task                                odd one out                    
measure         triplet accuracy dimensionality triplet accuracy   
observer                                                           
O1                          85.6            2.0             95.7   
O2                          93.2            2.0             97.3   
O3                          86.2            1.0             91.3   
O4                          75.8            1.0             95.1   
O5                          94.7            1.0             99.7   
average O1 - O5             87.1            1.4             95.8   
pooled O1 - O5              83.5            2.0             91.8   
P1                          92.8            1.0             99.7   
P2                          86.8            3.0             92.8   

image_set                 mean                                  \
task                   classic                     odd one out   
measure         dimensionality triplet accuracy dimensionality   
observer                                                         
O1                         2.3             84.9            1.7   
O2                         1.7             90.0            1.7   
O3                         1.7             78.6            1.7   
O4                         1.7             77.8   

In [17]:
summary_formatted = format_summary_table(summary)
summary_formatted

image_set            action                greebles               taxonomic  \
task                classic odd one out     classic odd one out     classic   
observer                                                                      
O1                 2 (87.1)    2 (96.3)    2 (82.1)    1 (90.0)    3 (85.6)   
O2                 1 (86.8)    1 (96.1)    2 (89.9)    2 (98.4)    2 (93.2)   
O3                 2 (80.3)    3 (83.6)    1 (69.3)    1 (76.7)    2 (86.2)   
O4                 2 (80.4)    2 (91.8)    2 (77.3)    2 (89.9)    1 (75.8)   
O5                 1 (91.2)    1 (99.4)    3 (81.6)    2 (90.2)    2 (94.7)   
average O1 - O5  1.6 (85.1)  1.8 (93.4)  2.0 (80.0)  1.6 (89.0)  2.0 (87.1)   
pooled O1 - O5     4 (85.6)    3 (89.7)    2 (75.5)    2 (82.0)    2 (83.5)   
P1                 2 (91.4)    2 (98.6)    2 (77.0)    1 (84.2)    2 (92.8)   
P2                 2 (81.6)    1 (85.0)    2 (83.1)    1 (90.6)    3 (86.8)   

image_set                          mean              
task            odd one out     classic odd one out  
observer                                             
O1                 2 (95.7)  2.3 (84.9)  1.7 (94.0)  
O2                 2 (97.3)  1.7 (90.0)  1.7 (97.3)  
O3                 1 (91.3)  1.7 (78.6)  1.7 (83.9)  
O4                 1 (95.1)  1.7 (77.8)  1.7 (92.2)  
O5                 1 (99.7)  2.0 (89.2)  1.3 (96.4)  
average O1 - O5  1.4 (95.8)  1.9 (84.1)  1.6 (92.8)  
pooled O1 - O5     2 (91.8)  2.7 (81.5)  2.3 (87.8)  
P1                 1 (99.7)  2.0 (87.1)  1.3 (94.2)  
P2                 3 (92.8)  2.3 (83.8)  1.7 (89.5)

In [18]:
# DISPLAY_IMAGE_SETS = {'action': 'Action', 'greebles': 'Greebles',
#                       'taxonomic': 'Taxonomic', 'mean': 'Mean'}
# DISPLAY_TASKS = {'classic': 'Classic', 'odd one out': 'Odd-one-out'}
# DISPLAY_OBSERVERS = {AVERAGE_OBSERVER_LABEL: 'Average', POOLED_OBSERVER: 'Pooled'}

# # row blocks, one per experiment: (label, observers in display order)
# EXPERIMENT_BLOCKS = [
#     ('Exp. 1', EXP1_OBSERVERS + [AVERAGE_OBSERVER_LABEL, POOLED_OBSERVER]),
#     ('Exp. 2', EXP2_OBSERVERS),
# ]
# # observers whose row is preceded by a rule separating them from the individual observers
# RULED_OBSERVERS = [AVERAGE_OBSERVER_LABEL]


# def summary_table_to_latex(formatted, caption, label):
#     """Write the formatted table as LaTeX, grouped into experiment blocks.

#     Follows the structure of sample_table.txt: two index columns (experiment, observer),
#     image sets as the outer column group with tasks nested inside, and \\multirow blocks
#     per experiment.
#     """
#     image_sets = list(dict.fromkeys(level for level, _ in formatted.columns))
#     n_columns = 2 + formatted.shape[1]
#     indent = ' ' * 4

#     # group header, one \multicolumn per image set, plus the rules underneath it
#     group_header = ' & '.join(
#         [' ', ' '] + [f'\\multicolumn{{{len(TASKS)}}}{{c}}{{{DISPLAY_IMAGE_SETS[image_set]}}}'
#                       for image_set in image_sets])
#     group_rules = ' '.join(
#         f'\\cmidrule(lr){{{3 + len(TASKS) * position}-{2 + len(TASKS) * (position + 1)}}}'
#         for position in range(len(image_sets)))
#     task_header = ' & '.join(
#         [' ', ' '] + [f'\\makecell{{{DISPLAY_TASKS[task]}}}'
#                       for _ in image_sets for task in TASKS])

#     lines = [
#         '\\begin{table}[h!]',
#         indent + '\\centering',
#         indent + f'\\begin{{tabular}}{{ll{"r" * formatted.shape[1]}}}',
#         indent + '\\toprule',
#         indent + group_header + ' \\\\',
#         indent + group_rules,
#         indent + task_header + ' \\\\',
#         indent + 'Experiment & Observer' + ' &' * formatted.shape[1] + ' \\\\',
#         indent + '\\midrule',
#     ]

#     for block_position, (experiment, observers) in enumerate(EXPERIMENT_BLOCKS):
#         if block_position > 0:
#             lines.append(indent + '\\noalign{\\vskip 3pt}'
#                          f'\\cline{{1-{n_columns}}}\\noalign{{\\vskip 3pt}}')
#         for row_position, observer in enumerate(observers):
#             if observer in RULED_OBSERVERS:
#                 lines.append(indent + f'\\cmidrule(lr){{2-{n_columns}}}')
#             first_column = (f'\\multirow[t]{{{len(observers)}}}{{*}}{{{experiment}}}'
#                             if row_position == 0 else ' ')
#             values = ' & '.join(formatted.loc[observer].tolist())
#             lines.append(indent + f'{first_column} & '
#                          f'{DISPLAY_OBSERVERS.get(observer, observer)} & {values} \\\\')

#     lines += [
#         indent + '\\bottomrule',
#         indent + '\\end{tabular}',
#         indent + f'\\caption{{{caption}}}',
#         indent + f'\\label{{{label}}}',
#         '\\end{table}',
#     ]
#     return '\n'.join(lines)


# latex_table = summary_table_to_latex(
#     summary_formatted,
#     caption=('Dimensionality of embeddings and triplet accuracy for different observers. '
#              'Each cell shows the estimated dimensionality followed by the triplet accuracy '
#              'in percent in parentheses. The average dimensionality is the average of the '
#              'individual embeddings, whereas the pooled dimensionality is the dimensionality '
#              'of one embedding created from pooling data of all five observers (O1 to O5). '
#              'The Mean columns average across the three image sets.'),
#     label='table:results_dimensionality_and_accuracy',
# )
# print(latex_table)